In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr


rng = np.random.default_rng(0)
n_families = 150
copies = 4
dim = 200

family_trait = rng.normal(size=(n_families, dim) ) # each family's "DNA", the dimensionality
X = np.repeat(family_trait, copies, axis = 0)    # number of near-copies per family
X = X + rng.normal(scale=0.05, size=X.shape)    # tiny differences
family_id = np.repeat(np.arange(n_families), copies)   # which family each belongs to

# target = strong per-family quirk + weak generalizable signal + noise
family_offset = rng.normal(scale=3.0, size=n_families)
weak_signal   = rng.normal(scale=0.3, size=dim)
y = family_offset[family_id] + X @ weak_signal + rng.normal(scale=1.0, size=len(X))

print(X.shape, y.shape)


(600, 200) (600,)


In [ ]:
n=len(X)
# LEAKY split: shuffle individuals randomly. Near-copies land on BOTH sides.
idx = rng.permutation(n)
test_leaky = idx[:120]; train_leaky = idx[120:]

# HONEST split: split by FAMILY. A whole family is test or train, never both.
test_families = rng.permutation(n_families)[:30]
test_honest = np.where(np.isin(family_id, test_families))[0]
train_honest = np.where(~np.isin(family_id, test_families))[0]

def score (train, test, label):
  m = Ridge().fit(X[train], y[train])
  rho = spearmanr(m.predict(X[test]), y[test]).correlation
  print(f"{label:8s} Spearman = {rho:.3f}")
  return rho

leaky  = score(train_leaky,  test_leaky,  "LEAKY")
honest = score(train_honest, test_honest, "HONEST")
print(f"\nInflation: {leaky - honest:+.3f}")

LEAKY    Spearman = 0.938
HONEST   Spearman = 0.381

Inflation: +0.557
